# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Mount Google Drive
Connect to Google Drive so you can load the dataset and save your results.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Run the cell below every time to activate the installed environment.

In [3]:
!pip uninstall -y torch torchvision torchaudio transformers protobuf tensorflow tensorflow-cpu
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q vllm==0.8.5 transformers==4.51.3 accelerate bitsandbytes tqdm sympy antlr4-python3-runtime==4.11.1
!pip install -q protobuf==4.25.3

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: protobuf 4.25.3
Uninstalling protobuf-4.25.3:
  Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xgrammar 0.1.18 requires transformers>=4.38.0, which is not installed.
compressed-tensors 0.9.3 requires transformers, which is not installed.
vllm 0.8.5 requires protob

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Please connect to a GPU runtime (Runtime > Change runtime type).")

NVIDIA A100-SXM4-80GB


In [5]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES (Changed from 1 to 0 for Colab)
DATA_PATH   = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/results/frq_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

INFO 05-31 00:25:37 [__init__.py:239] Automatically detected platform cuda.


In [59]:
DATA_PATH   = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/data/private.jsonl"
data = [json.loads(line) for line in open(DATA_PATH)]
len(data)

943

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [6]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [7]:
PROMPT_VARIANTS = {
    "concise": {
        "math": (
            "You are an expert mathematician. Use concise step-by-step reasoning. "
            "Avoid unnecessary explanation. "
            "Put your final answer inside \\boxed{}. "
            "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. Solve the problem concisely."
            "Choose the single best answer choice. "
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },

    "minimal_reasoning": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "End with the final answer inside \\boxed{}. "
            "If there are multiple answers, put them comma-separated inside one box, "
            "e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },

    "symbolic_exact": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "Final answer rules:"
            "- Use exact form."
            "- No decimal approximations."
            "- Keep expressions symbolic."
            "- Use \\frac{}{}, powers, \\sqrt{}, \\pi, \\ln{}, \\arctan{} when appropriate."
            "- Only use decimals for numbers that are already decimals in the problem."
            "- End with the final answer in \\boxed{...}."
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "examples": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "Examples of preferred style:"
            "- \\boxed{\\frac{5}{8}}, not \\boxed{0.625}"
            "- \\boxed{\\left(\\frac{1}{2}\\right)^{\\frac{1999-1963}{31}}}, not \\boxed{0.447}"
            "- \\boxed{\\arctan(4.76), \\pi}, not \\boxed{1.360, 3.142}"
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "multiple_answers": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "Final answer rules:"
            "- Include EVERY requested answer in the final answer."
            "- If the problem has multiple parts or multiple blanks, put all answers in ONE final \\boxed{...}, separated by commas."
            "- Use exact form."
            "- No decimal approximations."
            "- Keep expressions symbolic."
            "- Use \\frac{}{}, powers, \\sqrt{}, \\pi, \\ln{}, \\arctan{} when appropriate."
            "- Only use decimals for numbers that are already decimals in the problem."
            "- End with the final answer in \\boxed{...}."
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "combo": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "Final answer rules:"
            "- Include EVERY requested answer in the final answer."
            "- If the problem has multiple parts or multiple blanks, put all answers in ONE final \\boxed{...}, separated by commas."
            "- Use exact form."
            "- No decimal approximations."
            "- Keep expressions symbolic."
            "- Keep fractions, powers, roots, logarithms, and pi symbolic."
            "- Keep inverse trig answers symbolic: write \\arctan(x), \\arcsin(x), etc., not decimal approximations."
            "- Only use decimals for numbers that are already decimals in the problem."
            "- End with the final answer in \\boxed{...}."
            "Examples of preferred style:"
            "- \\boxed{\\frac{5}{8}}, not \\boxed{0.625}"
            "- \\boxed{\\left(\\frac{1}{2}\\right)^{\\frac{1999-1963}{31}}}, not \\boxed{0.447}"
            "- \\boxed{\\arctan(4.76), \\pi}, not \\boxed{1.360, 3.142}"
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
}


In [8]:
def build_prompt(
    question: str,
    options: Optional[list],
    variant: str = "baseline"
    )-> tuple[str, str]:

    """Return (system_prompt, user_prompt) for a question."""

    prompts = PROMPT_VARIANTS[variant]

    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return prompts['mcq'], f"{question}\n\nOptions:\n{opts_text}"
    return prompts['math'], question

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-31 00:26:04 [config.py:717] This model supports multiple tasks: {'generate', 'reward', 'score', 'embed', 'classify'}. Defaulting to 'generate'.
WARNING 05-31 00:26:04 [config.py:830] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-31 00:26:04 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=32768.
WARNING 05-31 00:26:06 [utils.py:2382] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 05-31 00:27:48 [core_client.py:439] Core engine process 0 ready.
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [10]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [11]:
# Build prompts for specific FRQ questions
frq_ids = [2, 3, 5, 7, 8, 16]
N = 150
#frq_data = [data[i] for i in frq_ids]
frq_data = data[:N]


In [12]:
# Build prompts for first N entries
prompts = {}
for variant in PROMPT_VARIANTS:
  prompts[variant] = []
  for item in frq_data:
    system, user = build_prompt(item["question"], item.get("options"), variant)
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts[variant].append(prompt_text)

In [13]:
responses = {}
finish_reasons = {}

for variant in PROMPT_VARIANTS:
    print(f"Generating responses for {len(prompts[variant])} questions for variant: {variant}...")

    outputs = llm.generate(prompts[variant], sampling_params=sampling_params)

    responses[variant] = [
        out.outputs[0].text.strip()
        for out in outputs
    ]

    finish_reasons[variant] = [
        out.outputs[0].finish_reason
        for out in outputs
    ]

Generating responses for 150 questions for variant: concise...


Processed prompts:   0%|          | 0/150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 150 questions for variant: minimal_reasoning...


Processed prompts:   0%|          | 0/150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 150 questions for variant: symbolic_exact...


...

Processed prompts:   0%|          | 0/150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 150 questions for variant: multiple_answers...


Processed prompts:   0%|          | 0/150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 150 questions for variant: combo...


Processed prompts:   0%|          | 0/150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [14]:
# Preview first question for each variant
for variant in PROMPT_VARIANTS:
#for i in range(min(3, len(responses))):
    i = 0
    print(f"\n── Response {i} (id={data[i].get('id')}), (variant: {variant}) ──")
    print(responses[variant][i])
    #print(responses[variant][i][:400], "..." if len(responses[variant][i]) > 400 else "")


── Response 0 (id=0), (variant: concise) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first positive even whole numbers are. The positive even whole numbers start from 2, 4, 6, 8, and so on. So the first one is 2, the second is 4, third is 6, etc. 

Wait, the problem says "the first 325 positive even whole numbers". So that's 2, 4, 6, ..., up to the 325th term. Let me recall that the nth even number is 2n. For example, the 1st even number is 2*1=2, the 2nd is 2*2=4, the 3rd is 2*3=6, so yeah, the kth even number is 2k. So the 325th even number is 2*325=650. 

So the sequence we're summing is 2, 4, 6, ..., 650. This is an arithmetic sequence where the first term a₁ is 2, the common difference d is 2, and the number of terms n is 325. 

The formula for the sum of the first n terms of an arithmetic sequence is Sₙ = n/2 * (a₁ + aₙ), where aₙ is the nth term. Alternatively, since it's the sum of even n

In [15]:
from collections import Counter

for variant in PROMPT_VARIANTS:
    print(f"\nVariant: {variant}")
    print(Counter(finish_reasons[variant]))


Variant: concise
Counter({'stop': 139, 'length': 11})

Variant: minimal_reasoning
Counter({'stop': 136, 'length': 14})

Variant: symbolic_exact
Counter({'stop': 136, 'length': 14})

Variant: examples
Counter({'stop': 137, 'length': 13})

Variant: multiple_answers
Counter({'stop': 138, 'length': 12})

Variant: combo
Counter({'stop': 139, 'length': 11})


In [16]:
# Compare answer lengths between the four variants

response_lengths = {} # To store lengths for each variant

for variant in PROMPT_VARIANTS:
    lengths = [len(response) for response in responses[variant]]
    response_lengths[variant] = lengths

print("Average Response Lengths per Variant:")
print("=" * 50)
for variant, lengths in response_lengths.items():
    if lengths:
        average_length = sum(lengths) / len(lengths)
        print(f"  {variant.ljust(15)}: {average_length:.2f} characters")
    else:
        print(f"  {variant.ljust(15)}: No responses generated.")
print("=" * 50)

Average Response Lengths per Variant:
  concise        : 13007.88 characters
  minimal_reasoning: 14176.07 characters
  symbolic_exact : 14468.95 characters
  examples       : 14874.84 characters
  multiple_answers: 13762.54 characters
  combo          : 14151.98 characters


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [17]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
import sys
sys.path.insert(0, "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition")
from judger import Judger
judger = Judger(strict_extract=False)

results = {}
for variant in PROMPT_VARIANTS:
  results[variant] = []
  for item, response in tqdm(zip(frq_data, responses[variant]), total=len(data), desc="Scoring"):
      is_mcq = bool(item.get("options"))
      gold   = item["answer"]

      if is_mcq:
          correct = score_mcq(response, str(gold))
      else:
          gold_list = gold if isinstance(gold, list) else [gold]
          try:
              correct = judger.auto_judge(
                  pred=response,
                  gold=gold_list,
                  options=[[]] * len(gold_list),
              )
          except Exception:
              correct = False

      results[variant].append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "gold":     gold,
          "response": response,
          "correct":  correct,
      })

#print(f"Scoring complete. {len(results)} results.")
print("Scoring complete")

Scoring:  13%|█▎        | 150/1126 [00:08<00:52, 18.45it/s]

Scoring complete


In [18]:
import pandas as pd
from judger import Judger

judger = Judger(strict_extract=False)

rows = []

for variant, result_list in results.items():
    for idx, r in enumerate(result_list):
        response = r["response"]
        extracted_raw = judger.extract_ans(response)

        if extracted_raw:
            extracted_split = judger.split_by_comma(extracted_raw)
            extracted_normalized = [
                judger.norm_ans_str(ans)
                for ans in extracted_split
            ]
        else:
            extracted_split = []
            extracted_normalized = []

        rows.append({
            "variant": variant,
            "idx": idx,
            "id": r.get("id"),
            "is_mcq": r.get("is_mcq"),
            "gold": r.get("gold"),
            "correct": r.get("correct"),
            "response": response,
            #"judger_extracted_raw": extracted_raw,
            #"judger_extracted_split": extracted_split,
            "judger_extracted_normalized": extracted_normalized,
            "response_chars": len(response) if isinstance(response, str) else None,
        })

debug_df = pd.DataFrame(rows)
debug_df.head(2)

,variant,idx,id,is_mcq,gold,correct,response,judger_extracted_normalized,response_chars
0,concise,0,0,False,[325*(1+325)],True,"Okay, let's see. I need to find the sum of the...",[105950],3373
1,concise,1,1,True,F,False,"Okay, let's try to solve this integral. The pr...",[E],21883


In [26]:
debug_df[(debug_df["variant"] == "multiple_answers") & (debug_df["correct"] == False) & (debug_df["is_mcq"] == False)]

,variant,idx,id,is_mcq,gold,correct,response,judger_extracted_normalized,response_chars
602,multiple_answers,2,2,False,"[143.224229233795, 2.32624773420025]",False,"Okay, let's try to solve this problem. It's ab...","[75+\frac{160\sqrt{22}{}}{11}, \frac{30\ln(\fr...",20160
612,multiple_answers,12,12,False,"[380, 315, 13, 310]",False,"Okay, let's tackle these problems one by one. ...","[380, 315, 14, 310]",5321
616,multiple_answers,16,16,False,"[atan(4.76), pi]",False,"Okay, let's see. The problem is to find all so...","[1.364, 3.142]",15864
620,multiple_answers,20,20,False,"[-10, 100, -9, 81, 1, 1, 3, 9, 7, 49, 8, 64, 3...",False,"Okay, let's try to figure out how to find the ...",[7.797],10293
621,multiple_answers,21,21,False,"[3*t^1*(1-t)^2, 6*t^2*(1-t)^2, 10*t^3*(1-t)^2,...",False,"Okay, let's try to figure out the Bernstein po...","[3t(1-t)^2, 6t^2(1-t)^2, 10t^3(1-t)^2, 7t^6(1-...",15076
622,multiple_answers,22,22,False,"[1.6, 1.76, 0.16*p/16, up, 1, 0.16]",False,"Okay, let's tackle this problem step by step. ...","[1.60, 1.76, 0.16\lceil\frac{p}{16}\rceil, up,...",32051
625,multiple_answers,25,25,False,"[18.8105, 11.3449, Yes]",False,"Okay, let's tackle this problem step by step. ...","[\frac{1787}{95}, 11.34, True]",20255
627,multiple_answers,27,27,False,[[ln(0.5)]/[ln(0.96584)]],False,"Okay, let's see. I need to find the half-life ...",[19.94],18401
634,multiple_answers,34,34,False,[14.5843461351483],False,"Okay, let's see. The problem says there's a ci...",[\frac{504}{11\pi}],3389
637,multiple_answers,37,37,False,"[110101, 11010101, 1010100001]",False,"Okay, let's tackle these binary addition probl...","[110101(asweverifiedwithdecimal), andforthesec...",33784


In [57]:
debug_df[debug_df["id"] == 16].get('response').values[-1]

'Okay, let\'s try to figure out this problem. The question is to determine all solutions for tan(θ) = 4.76, and the solutions are supposed to be θ = [ANS] + [ANS]n where n is any integer, and the first blank is between -π/2 and π/2. \n\nFirst, I remember that the tangent function has a period of π, so the general solution for tan(θ) = k is θ = arctan(k) + nπ, where n is any integer. The arctan function gives the principal value, which is in the interval (-π/2, π/2), so that\'s exactly the first blank here. \n\nSo, the first blank should be arctan(4.76), and the second blank is π (since the period is π). But let me confirm. Let\'s recall that for tan(θ) = c, the solutions are θ = arctan(c) + nπ, n ∈ ℤ. So yes, the general solution is θ = arctan(4.76) + nπ. \n\nThe problem says "the value in each first blank lies between -π/2 and π/2". So the first blank is arctan(4.76), which is in (-π/2, π/2) because arctan\'s range is (-π/2, π/2). \n\nNow, we need to compute arctan(4.76) and express i

## 8. Summary

Print accuracy broken down by question type.

In [20]:
for variant in PROMPT_VARIANTS:
  print(f"Variant: {variant}")
  mcq_res  = [r for r in results[variant] if r["is_mcq"]]
  free_res = [r for r in results[variant] if not r["is_mcq"]]

  def acc(subset):
      return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

  print("=" * 50)
  print("EVALUATION RESULTS")
  print("=" * 50)
  print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
  print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
  print(f"  Overall    : {sum(r['correct'] for r in results[variant]):4d} / {len(results[variant]):4d}  ({acc(results[variant]):.2f}%)")
  print("=" * 50)

Variant: concise
EVALUATION RESULTS
  MCQ        :   44 /   56  (78.57%)
  Free-form  :   53 /   94  (56.38%)
  Overall    :   97 /  150  (64.67%)
Variant: minimal_reasoning
EVALUATION RESULTS
  MCQ        :   44 /   56  (78.57%)
  Free-form  :   50 /   94  (53.19%)
  Overall    :   94 /  150  (62.67%)
Variant: symbolic_exact
EVALUATION RESULTS
  MCQ        :   40 /   56  (71.43%)
  Free-form  :   41 /   94  (43.62%)
  Overall    :   81 /  150  (54.00%)
Variant: examples
EVALUATION RESULTS
  MCQ        :   42 /   56  (75.00%)
  Free-form  :   42 /   94  (44.68%)
  Overall    :   84 /  150  (56.00%)
Variant: multiple_answers
EVALUATION RESULTS
  MCQ        :   40 /   56  (71.43%)
  Free-form  :   57 /   94  (60.64%)
  Overall    :   97 /  150  (64.67%)
Variant: combo
EVALUATION RESULTS
  MCQ        :   44 /   56  (78.57%)
  Free-form  :   56 /   94  (59.57%)
  Overall    :  100 /  150  (66.67%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [21]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

TypeError: string indices must be integers, not 'str'

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!